In [ ]:
import os
import sys
from os import path

import time
import logging
import argparse
import re
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import pandas as pd
import math
from statistics import mean
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import Draw
import zipfile
from jinja2 import Environment, PackageLoader
import numpy as np
import matplotlib.cm as cm
import glob

import getWatersData
import cpAssignment
import genOutput
import plotting

In [ ]:
def getUserReadableWell(wellno, plate_col_no):
    """
    Converts the well as a number into a user-friendly string,
    e.g. well 11 becomes "B5" for a 4*6 well plate. 

    :param wellno: An integer representing a specific well on the plate
    
    :return: A string representing a specific well on the plate
    """
    
    rowVal = math.floor((wellno-1) / plate_col_no)
    colVal = (wellno) % plate_col_no
    if colVal == 0:
        colVal = plate_col_no
    
    label = f'{chr(ord("@")+(rowVal)+1)}{colVal}'
    return label

def generateMol(smiles, name, save_dir):
    """
    Generates a 2D rendering of the given structure and saves 
    it as a .png

    :param smiles: a string (SMILES) of the compound
    :param name: a string corresponding to name of that compound
    :param save_dir: a string corresponding to the output directory
    
    :return: A .png rendering of the given compound
    """

    mol = Chem.MolFromSmiles(smiles.strip())
    _discard = AllChem.Compute2DCoords(mol)
    Draw.MolToFile(mol, f'{save_dir}structures/{name}.png', size=(200, 150))

def buildHTML(save_dir, cpTable, analysis_name, times = {}):
    """
    Build a HTML output file using jinja2 and a html_template
    that is stored in the directory "templates". 
    
    :param save_dir: A string designating the output directory
    :param cpTable: Pandas datatable containing all information on 
                        the compounds used for analysis
    :param all_compounds: a list of all compound names
    :param impurities: a list of all impurity names
    :param analysis_name: User provided name for the analysis
    :param times: Optional parameter of a list of floats related to processing time
                    for each step of the analysis. 
    :return: HTML file saved to save_dir 
    """


    
    env = Environment(
        loader = PackageLoader("PyParse", "templates")
    )

    template = env.get_template("html_template.html")

    cptablerows = cpTable.to_dict(orient="records")
    

    with open(f'{save_dir}/html_output.html', "w") as fo:
        fo.write(template.render(
            cpnames = list(cpTable.loc[cpTable["type"] != "Impurity", "name"]),
            imp_no = len(cpTable.loc[cpTable["type"] == "Impurity"].index),
            cptablerows = cptablerows, 
            save_dir = save_dir,
            path = path,
            times = times,
            round = round,
            pt = options.plot_type, 
            analysis_name = analysis_name,
            options = vars(options)
            )
        )
    fo.close() 

In [ ]:
#Create the save directory if one isn't already present.
save_dir = "C:/Users/nosam/OneDrive/Documents/GitHub/PyParse/testing/Performance Testing/"

error_msg = ""
try:
    os.mkdir(save_dir)
except OSError as error:
    error_msg = f'The directory "{save_dir}" already exists.'

    #make sub-directories to store all graphs and pictures of structures
try: 
    os.mkdir(f'{save_dir}/graphs')
except OSError as error:
    logging.debug("Graphs directory already exists.")
try: 
    os.mkdir(f'{save_dir}/structures')
except OSError as error:
    logging.debug("Structures directory already exists.")

In [ ]:
rawData = getWatersData.rawWatersData("example_dataset/Waters/Example2/LC-MS Data for 48-Well Plate.rpt")

#Determine how the file records the well (e.g. "A,1", 1, "1,1", etc)
#Note that this must be done before processing the data
rawData.getWellFormat() 

#Process the data types in turn
rawData.processDAD()
rawData.processMS()
rawData.processUV()
rawData.processTrace()
rawData.processELSD()

#Select the lcData we want to use for this analysis and trim the traceData accordingly 

msData = rawData.rawMSTable
uvData = rawData.rawUVTable

In [ ]:
cpTable = cpAssignment.Assignment("example_dataset/Waters/Example2/PyParse_designer_platemap.csv", 12, 8)
cpTable.generateCPTable()
cpTable.generateEMs("True") #Generate exact masses for each compound

cpTable.cpTable["type"] = pd.Categorical(cpTable.cpTable["type"], ["Product", "Reactant", "InternalSTD", "Byproduct"])
cpTable.cpTable.sort_values("type", inplace=True)

In [ ]:
#Find hits for each compound
cpTable.findHits(msData, lcData)

In [ ]:
cpTable.validateHits(lcData, msData, uvData)

In [ ]:
cpTable.removeDupAssigns(lcData, "mass_conf")

In [ ]:
cpTable.findImpurities(lcData, msData, save_dir)

In [ ]:
output = genOutput.Output(rawData.sample_IDs, 12, 8)
output.generateOutputTable(cpTable.cpTable, lcData)

In [ ]:
cpTable.setBestWell(output.df, lcData, "corrP/STD")
cpTable.setBestMS(lcData)
cpTable.setBestTime(lcData)
cpTable.setBestPurity(lcData)

In [ ]:
cpTable.findPotentialConflicts()

In [ ]:
plotting.plotHeatmaps(output.df, save_dir, 12, 8)

In [ ]:
byproducts = cpTable.cpTable.loc[cpTable.cpTable["type"] == "Byproduct", "name"]
plotting.plotPieCharts(options.plot_type, output.df, save_dir, byproducts, 12, 8)

In [ ]:
cpTable.cpTable.apply(plotting.plotHitValidationGraph, args = (lcData, save_dir, 12, 8,), axis = 1)

In [ ]:
cpTable.cpTable.apply(plotting.plotChroma, args=(cpTable.cpTable, lcData, msData, 
                                                 traceData, save_dir,), axis = 1)

In [ ]:
plotting.plotHistogram(output.df, save_dir)
plotting.plotDonut(output.df, save_dir)

#Generate a set of PNG files to depict each compound
cpTable.cpTable.apply(lambda row: generateMol(row["canonSMILES"], row["name"], save_dir), axis = 1)

#Generate a location map for each compound
cpTable.cpTable.apply(plotting.genLocationHeatmaps, args=(save_dir, 12, 8,), axis = 1)


In [ ]:
cpTable.findOverlap(lcData)

In [ ]:
#build the HTML output file
buildHTML(save_dir, cpTable.cpTable, "", times = times)